# Etapa 8 — Ajuste con más países de RDD2022 (Tesla T4)
Los dos modelos parten de su `best.pt` de la etapa 7 y se ajustan 50 épocas con RDD2022 China (train y val de la etapa 7) más fotos tomadas desde auto de Japón, India, República Checa y Estados Unidos (`configs/stage8_paises.yaml`). El TEST de China y el 10 % de test de los países nuevos quedan bloqueados: no se evalúan aquí.

Orden: celdas 1 a 5, **PREPARAR** (solo la primera vez), **LÍNEA BASE**, **SMOKE**, y recién después el entrenamiento. Si Colab se desconecta, vuelva a ejecutar las celdas 1 a 5 y luego la celda **Reanudar** del modelo que se cortó.

In [ ]:
# 1. Comprobar GPU asignada
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Seleccione Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0).upper()

In [ ]:
# 2. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Extraer el paquete en almacenamiento rápido temporal
from pathlib import Path
import shutil, zipfile
ZIP_PATH = Path('/content/drive/MyDrive/TP1_LMD/stage8/packages/colab_stage6.zip')
PACKAGE_ROOT = Path('/content/colab_stage6')
RESULTS_ROOT = Path('/content/drive/MyDrive/TP1_LMD/stage8/results')
CONFIG = PACKAGE_ROOT / 'configs' / 'stage8_paises.yaml'
STAGE7 = Path('/content/drive/MyDrive/TP1_LMD/stage7/results/training')
assert ZIP_PATH.is_file(), ZIP_PATH
for name in ('lmd_yolo', 'yolov8n'):
    assert (STAGE7 / name / 'weights' / 'best.pt').is_file(), f'Falta el best.pt de la etapa 7 de {name}'
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')
assert CONFIG.is_file()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(PACKAGE_ROOT, RESULTS_ROOT, CONFIG)

In [ ]:
# 4. Verificar todos los hashes del paquete
import subprocess, sys
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/manifests.py'), 'verify-package'], check=True)

In [ ]:
# 5. Instalar dependencias generales; NO instala ultralytics ni reemplaza torch CUDA
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE_ROOT/'requirements-colab.txt')], check=True)

In [ ]:
# 6. PREPARAR (solo la primera vez): descarga los países, convierte a YOLO, divide 80/10/10 y guarda el dataset en Drive.
# Los ZIP de cada país quedan en MyDrive/TP1_LMD/rdd2022 para no volver a descargarlos.
if not (RESULTS_ROOT / 'dataset_stage8_manifest.json').exists():
    subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'prepare', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)
else:
    print('El dataset ya estaba preparado; se restaura desde Drive')
    subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'restore', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 7. LÍNEA BASE: modelos de la etapa 7 sobre la validación nueva (China y países por separado)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'baseline', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 8. SMOKE: 1 época sobre 2 % de TRAIN para cada modelo
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'smoke', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)
import json
print(json.dumps(json.loads((RESULTS_ROOT/'smoke_gate.json').read_text()), indent=2))
print('STOP: revise el smoke antes de entrenar.')

## Gate manual para las 50 épocas
Cambie la variable a `True` solo después de revisar el smoke.

In [ ]:
RUN_LONG_TRAINING = False  # cambiar manualmente a True tras revisar el smoke
assert RUN_LONG_TRAINING, 'Gate cerrado: no se lanzará el entrenamiento automáticamente.'

In [ ]:
# 9A. LMD_YOLO: 50 épocas desde el best.pt de la etapa 7
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'train', '--model', 'lmd_yolo', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9A-R. Reanudar LMD_YOLO si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'train', '--model', 'lmd_yolo', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9B. YOLOv8n: 50 épocas desde el best.pt de la etapa 7 (referencia del paper)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'train', '--model', 'yolov8n', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9B-R. Reanudar YOLOv8n si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'train', '--model', 'yolov8n', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 10. Comparación etapa 7 vs etapa 8 y curvas (requiere los dos modelos); no usa TEST
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage8.py'), 'compare', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)